<center>

# Predictor de Tiempo de Hospitalizacion 

</center>

In [61]:
import pandas as pd
import numpy as np

## Cargar datos

In [62]:
# Cargar dataSet
df_healthcare = pd.read_csv('healthcare_dataset.csv')

In [63]:
# Convertir las columnas a tipo fecha
df_healthcare['Date of Admission'] = pd.to_datetime(df_healthcare['Date of Admission'])
df_healthcare['Discharge Date'] = pd.to_datetime(df_healthcare['Discharge Date'])

# Crear la variable objetivo (duración en días)
df_healthcare['Length of Stay'] = (df_healthcare['Discharge Date'] - df_healthcare['Date of Admission']).dt.days

# Cargar de las primeras 5 columnas del archivo
df_healthcare.head()

,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results,Length of Stay
0,Bobby JacksOn,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.281306,328,Urgent,2024-02-02,Paracetamol,Normal,2
1,LesLie TErRy,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327287,265,Emergency,2019-08-26,Ibuprofen,Inconclusive,6
2,DaNnY sMitH,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook PLC,Aetna,27955.096079,205,Emergency,2022-10-07,Aspirin,Normal,15
3,andrEw waTtS,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers and Vang,",Medicare,37909.782410,450,Elective,2020-12-18,Ibuprofen,Abnormal,30
4,adrIENNE bEll,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-White,Aetna,14238.317814,458,Urgent,2022-10-09,Penicillin,Abnormal,20


## Preparar conjuntos de entrenamientos y pruebas

In [64]:
# Categorizar el campo income_cat para estratificar los sets de training y test
df_healthcare["age_cat"] = pd.cut(df_healthcare["Age"],
                                    bins=[0,18,35,50,65,np.inf],
                                    labels=[1, 2, 3, 4, 5]) 

In [65]:
# Generando los dataSets estratificados
 
from sklearn.model_selection import train_test_split
 
strat_train_set, strat_test_set = train_test_split(df_healthcare, 
                                                   test_size=0.2, 
                                                   stratify=df_healthcare["age_cat"], 
                                                   random_state=42)
 
print('Tamaño del train_set',strat_train_set.shape)
print('Tamaño del test_set',strat_test_set.shape)

Tamaño del train_set (44400, 17)
Tamaño del test_set (11100, 17)


In [66]:
# Prepara el dataSet de training
# Separar los atributos de la variable objetivo
 
dfDistTrain = strat_train_set.drop("Length of Stay", axis=1)
dfDistTrain.drop('age_cat', axis=1, inplace=True)
dfDistTrain_labels = strat_train_set["Length of Stay"].copy()
 
print(dfDistTrain.shape)
print(dfDistTrain_labels.shape)

(44400, 15)
(44400,)


In [67]:
import numpy as np

# Copia del conjunto predictivo (para no alterar el original)
dfDistTrain_nulos = dfDistTrain.copy()

# Cantidad de valores nulos a insertar
n = 800
np.random.seed(42)

# Columnas donde insertar nulos (algunas numéricas y categóricas)
columnas_con_nulos = ['Billing Amount', 'Medication', 'Test Results']

for col in columnas_con_nulos:
    idx = np.random.choice(dfDistTrain_nulos.index, n, replace=False)
    dfDistTrain_nulos.loc[idx, col] = np.nan

# Confirmar cuántos nulos hay
dfDistTrain_nulos.isnull().sum()


Name                    0
Age                     0
Gender                  0
Blood Type              0
Medical Condition       0
Date of Admission       0
Doctor                  0
Hospital                0
Insurance Provider      0
Billing Amount        800
Room Number             0
Admission Type          0
Discharge Date          0
Medication            800
Test Results          800
dtype: int64

In [68]:
# IMPUTACIÓN DE VALORES NULOS (adaptado al dataset actual)

from sklearn.impute import SimpleImputer
import numpy as np

# Se copia el DataFrame con nulos
dfDistTrain_imputado = dfDistTrain_nulos.copy()

# Imputación para variables numéricas (usando la mediana)
imputer_median = SimpleImputer(strategy='median')

# Seleccionamos solo las numéricas
num_cols = dfDistTrain_imputado.select_dtypes(include=[np.number]).columns

# Aplicamos el imputador
dfDistTrain_imputado[num_cols] = imputer_median.fit_transform(dfDistTrain_imputado[num_cols])

# Imputación para variables categóricas (usando la moda)
imputer_mode = SimpleImputer(strategy='most_frequent')

# Seleccionamos las categóricas
cat_cols = dfDistTrain_imputado.select_dtypes(exclude=[np.number]).columns

# Aplicamos el imputador
dfDistTrain_imputado[cat_cols] = imputer_mode.fit_transform(dfDistTrain_imputado[cat_cols])

# Verificamos que no haya nulos
dfDistTrain_imputado.isnull().sum()


Name                  0
Age                   0
Gender                0
Blood Type            0
Medical Condition     0
Date of Admission     0
Doctor                0
Hospital              0
Insurance Provider    0
Billing Amount        0
Room Number           0
Admission Type        0
Discharge Date        0
Medication            0
Test Results          0
dtype: int64

## Preparar transformadores y pipelines para procesamiento

In [69]:
# Bibliotecas para crear el transformador de clusters
 
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.cluster import KMeans

In [70]:
class ClusterSimilarity(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=10, gamma=1.0, random_state=None):
        self.n_clusters = n_clusters
        self.gamma = gamma
        self.random_state = random_state
 
    def fit(self, X, y=None, sample_weight=None):
        self.kmeans_ = KMeans(self.n_clusters, n_init=10,
                              random_state=self.random_state)
        self.kmeans_.fit(X, sample_weight=sample_weight)
        return self  
 
    def transform(self, X):
        return rbf_kernel(X, self.kmeans_.cluster_centers_, gamma=self.gamma)
    def get_feature_names_out(self, names=None):
        return [f"Cluster {i} similarity" for i in range(self.n_clusters)]

In [71]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.compose import make_column_selector

In [72]:
# Hacemos una copia del dataset imputado
dfTrain_final = dfDistTrain_imputado.copy()

# Aseguramos que las fechas estén en formato datetime
dfTrain_final["Date of Admission"] = pd.to_datetime(dfTrain_final["Date of Admission"])
dfTrain_final["Discharge Date"] = pd.to_datetime(dfTrain_final["Discharge Date"])

# Creamos la variable objetivo (no se queda en X)
dfTrain_final["Length of Stay"] = (
    dfTrain_final["Discharge Date"] - dfTrain_final["Date of Admission"]
).dt.days

# Definimos la variable objetivo
y = dfTrain_final["Length of Stay"].copy()

# Columnas que NO queremos usar como predictoras
cols_to_drop = [
    "Name",
    "Date of Admission",
    "Discharge Date",
    "Doctor",
    "Hospital",
    "Length of Stay"
]

# Creamos X sin la variable objetivo ni columnas irrelevantes
X = dfTrain_final.drop(columns=cols_to_drop)

print("Columnas en X:", X.columns.tolist())
print("Variable objetivo (y):", y.name)

Columnas en X: ['Age', 'Gender', 'Blood Type', 'Medical Condition', 'Insurance Provider', 'Billing Amount', 'Room Number', 'Admission Type', 'Medication', 'Test Results']
Variable objetivo (y): Length of Stay


In [73]:
# Función para calcular las columnas nuevas (ratios)
def column_ratio(X):
    X = np.asarray(X, dtype=float)
    numerador = X[:, [0]]      # Billing Amount
    denominador = X[:, [1]]    # Length of Stay

    # Evitar división entre 0 o valores negativos
    denominador_seguro = np.where(denominador <= 0, 1e-6, denominador)

    return numerador / denominador_seguro
 
# Función para asignar los nombres de las nuevas columnas (ratios)
def ratio_name(function_transformer, feature_names_in):
    return ["ratio"]  # feature names out
 
# Pipeline para imputar las columnas sobre las que se basarán los ratios, luego genera los ratios 
# y finalmente estandariza los resultados
def ratio_pipeline():
    return make_pipeline(SimpleImputer(strategy="median"),
                         FunctionTransformer(column_ratio, feature_names_out=ratio_name),
                         StandardScaler())

def safe_log(X):
    X = np.asarray(X, dtype=float)
    # mínimo por columna
    mins = np.nanmin(X, axis=0)
    # para columnas con min <= 0, calculamos cuánto hay que sumar para hacerlas positivas
    offsets = np.where(mins <= 0, -mins + 1e-6, 0.0)
    X_shifted = X + offsets   # se suma por columna
    return np.log(X_shifted)


# Pipeline para imputar, aplicar una transformación logarítmica y estandarizar los resultados 
log_pipeline = make_pipeline(SimpleImputer(strategy="median"),
                             FunctionTransformer(safe_log, feature_names_out="one-to-one"),
                             StandardScaler())
 
# Función para calcular las similitudes con los 10 clusters
cluster_simil = ClusterSimilarity(n_clusters=10, gamma=1., random_state=42)
 
# Pipeline para imputar y estandarizar
default_num_pipeline = make_pipeline(SimpleImputer(strategy="median"),
                                     StandardScaler())

# Lista explícita de categóricas que SÍ queremos usar
cat_features = [
    "Gender",
    "Blood Type",
    "Medical Condition",
    "Insurance Provider",
    "Admission Type",
    "Medication",
    "Test Results"
]

# Pipeline: imputa y codfica categorías
cat_pipeline = make_pipeline(SimpleImputer(strategy="most_frequent"),
                             OneHotEncoder(handle_unknown="ignore"))
 
# Integra los pipelines
preprocessing = ColumnTransformer([("billing_per_age", ratio_pipeline(), ["Billing Amount", "Age"]),                                   
                                   ("log", log_pipeline, ["Billing Amount", "Age"]),
                                   ("cat", cat_pipeline, cat_features),
                                   ("num_rest", default_num_pipeline, ["Room Number"]),
                                  ],
                                  remainder="drop")  # evita que fechas caigan al pipeline numérico

In [74]:
# Aplica 
X_preproc = preprocessing.fit_transform(X)
print(type(X_preproc))
print("Shape transformada:", X_preproc.shape)
print("Cantidad de columnas resultantes:", len(preprocessing.get_feature_names_out()))

<class 'numpy.ndarray'>
Shape transformada: (44400, 36)
Cantidad de columnas resultantes: 36


In [75]:
from pandas import DataFrame

dfDistTrainProc = pd.DataFrame(
    X_preproc,
    columns=preprocessing.get_feature_names_out(),
    index=X.index
)

dfDistTrainProc.head(2)

,billing_per_age__ratio,log__Billing Amount,log__Age,cat__Gender_Female,cat__Gender_Male,cat__Blood Type_A+,cat__Blood Type_A-,cat__Blood Type_AB+,cat__Blood Type_AB-,cat__Blood Type_B+,...,cat__Admission Type_Urgent,cat__Medication_Aspirin,cat__Medication_Ibuprofen,cat__Medication_Lipitor,cat__Medication_Paracetamol,cat__Medication_Penicillin,cat__Test Results_Abnormal,cat__Test Results_Inconclusive,cat__Test Results_Normal,num_rest__Room Number
40666,0.439625,1.145786,0.588217,1.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.471719
48983,-1.150262,-1.862861,0.840192,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,-1.519785


## SEMANA 11 ->

### Entrenar varios modelos y seleccionar uno

#### Regresión Lineal

#### Árbol de Decisión

#### Bosque Aleatorio

### Afinar modelo seleccionado

Aquí va lo que se eligió, lo de abajo va en el grande (pruebas)

Se afinará el modelo de Bosque Aleatorio dado que tuvo el mejor rendimiento.
[Bosque Aleatorio](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html) tiene varios hiperparámetros que veremos más adelante, por ahora se manipulará solo max_features.

#### Usando Grid Search

#### Usando Randomize Search